# ⚖️ Modul 03: Analisis Komparasi Head-to-Head & Benchmarking Arsitektur
### Evaluasi Komparatif Arsitektur Deep Learning: RNN (BiGRU) vs Transfer Learning (IndoBERT)
> **Tugas Individu Mata Kuliah Deep Learning**  
> **Nama Mahasiswa:** Haris  
> **NIM:** H071241070  
> **Program Studi:** Sistem Informasi, FMIPA Universitas Hasanuddin  

---

### 📌 Ringkasan Komparasi:
Modul ini mengomparasikan secara empiris kinerja seluruh model yang telah dilatih pada data uji independen yang sama persis (1.508 data):
1. **Baseline Machine Learning:** Linear SVM + TF-IDF N-Gram
2. **Arsitektur Deep Learning 1:** Recurrent Neural Network (Bidirectional GRU)
3. **Arsitektur Deep Learning 2:** Transfer Learning Transformer (Fine-Tuned IndoBERT-base-p1)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

# Tabel Rekapitulasi Data Kinerja
comparison_data = {
    'Metrik Pengujian': [
        'Akurasi Global',
        'Macro F1-Score (Evaluasi Utama)',
        'Weighted F1-Score',
        'F1-Score: Jalan Rusak (Minoritas Terkecil)',
        'F1-Score: Drainase (Minoritas)',
        'F1-Score: Lampu Jalan (Mayoritas 70%)',
        'F1-Score: Pangkas Pohon',
        'F1-Score: Sampah',
        'Rata-rata Latensi Inferensi',
        'Jumlah Parameter Model',
        'Kebutuhan Komputasi'
    ],
    'Baseline (Linear SVM)': [
        '98.01%', '95.39%', '98.00%', '90.06%', '91.91%', '99.48%', '97.62%', '97.89%',
        '0.85 ms / aduan', '10.000 bobot', 'Sangat Ringan (CPU)'
    ],
    'Arsitektur 1: RNN (BiGRU)': [
        '97.88%', '94.68%', '97.85%', '91.15%', '92.15%', '99.12%', '96.80%', '94.20%',
        '3.42 ms / aduan', '940.677 params', 'Ringan (GPU/CPU)'
    ],
    'Arsitektur 2: Transfer Learning (IndoBERT)': [
        '98.01%', '95.70%', '98.01%', '93.41%', '93.53%', '99.29%', '98.39%', '93.88%',
        '12.36 ms / aduan', '~124.5 Juta params', 'Sedang (Akselerasi GPU)'
    ],
    'Keunggulan / Trade-Off': [
        'Setara (98.01%)',
        'IndoBERT Terbaik (+0.31% vs SVM, +1.02% vs BiGRU)',
        'Setara di seluruh model (~98%)',
        'IndoBERT Unggul Signifikan (+3.35% vs SVM)',
        'IndoBERT Unggul (+1.62% vs SVM)',
        'Seluruh model menguasai kelas mayoritas (>99%)',
        'IndoBERT Terbaik (98.39%)',
        'Baseline SVM Sedikit Lebih Presisi',
        'SVM tercepat (0.85 ms), IndoBERT 12.36 ms (Real-Time)',
        'BiGRU efisien (<1M), IndoBERT representatif (124M)',
        'IndoBERT cocok untuk server cloud; BiGRU cocok untuk edge/CPU'
    ]
}

comp_df = pd.DataFrame(comparison_data)
display(comp_df)

## Visualisasi Komparasi Macro F1-Score & Akurasi

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5), dpi=120)

models = ['Baseline SVM', 'RNN (BiGRU)', 'Transfer Learning\n(IndoBERT)']
macro_f1 = [95.39, 94.68, 95.70]
colors = ['#94A3B8', '#38BDF8', '#2563EB']

bars = ax.bar(models, macro_f1, color=colors, width=0.45, edgecolor='#0F172A', linewidth=1.2)
ax.set_title("Evaluasi Metrik Utama: Macro F1-Score Antar-Arsitektur", fontsize=12, fontweight='bold', pad=12)
ax.set_ylabel("Macro F1-Score (%)", fontsize=10, fontweight='bold')
ax.set_ylim([90, 98])

for b in bars:
    y = b.get_height()
    ax.text(b.get_x() + b.get_width()/2.0, y + 0.25, f"{y:.2f}%", ha='center', va='bottom', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

## Analisis Kinerja pada Kelas Minoritas (Ketahanan terhadap Class Imbalance)

In [ ]:
cats = ['Drainase (9.3%)', 'Pangkas Pohon (8.3%)', 'Sampah (6.4%)', 'Jalan Rusak (5.6%)']
svm_scores = [91.91, 97.62, 97.89, 90.06]
bigru_scores = [92.15, 96.80, 94.20, 91.15]
indobert_scores = [93.53, 98.39, 93.88, 93.41]

x = np.arange(len(cats))
width = 0.26

fig, ax = plt.subplots(figsize=(10, 5), dpi=120)
ax.bar(x - width, svm_scores, width, label='Baseline SVM', color='#94A3B8')
ax.bar(x, bigru_scores, width, label='RNN (BiGRU)', color='#38BDF8')
ax.bar(x + width, indobert_scores, width, label='Transfer Learning (IndoBERT)', color='#2563EB')

ax.set_title("Perbandingan F1-Score pada 4 Kelas Minoritas Ekstrem", fontsize=12, fontweight='bold', pad=10)
ax.set_xticks(x)
ax.set_xticklabels(cats, fontsize=9.5)
ax.set_ylabel("F1-Score (%)", fontsize=10, fontweight='bold')
ax.set_ylim([85, 101])
ax.legend(loc='lower left', fontsize=9.5)

plt.tight_layout()
plt.show()

## Analisis Waktu Inferensi (Latency Benchmark)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4), dpi=120)

devices = ['SVM (CPU)', 'BiGRU (GPU)', 'IndoBERT (GPU)']
latencies = [0.85, 3.42, 12.36]
lat_colors = ['#10B981', '#F59E0B', '#EF4444']

bars = ax.bar(devices, latencies, color=lat_colors, width=0.45, edgecolor='#0F172A', linewidth=1)
ax.set_title("Perbandingan Rata-rata Latensi Inferensi per Aduan", fontsize=12, fontweight='bold', pad=10)
ax.set_ylabel("Latensi (ms / aduan)", fontsize=10, fontweight='bold')
ax.set_ylim([0, 15])

for b in bars:
    y = b.get_height()
    ax.text(b.get_x() + b.get_width()/2.0, y + 0.3, f"{y:.2f} ms", ha='center', va='bottom', fontweight='bold', fontsize=10)

ax.axhline(y=200, color='red', linestyle='--', alpha=0.3)
ax.text(0.5, 13.5, "Batas Maksimum Toleransi Web Real-Time (<200 ms)", color='#DC2626', fontsize=8.5, fontstyle='italic')

plt.tight_layout()
plt.show()

## 💡 Kesimpulan & Diskusi Ilmiah Tugas Deep Learning:
1. **Keunggulan Transfer Learning (IndoBERT):**
   * Meraih **Macro F1-Score tertinggi (95,70%)**, terutama menunjukkan lompatan performa paling signifikan pada kelas paling minoritas (*Jalan Rusak* mencapai **93,41%**, naik **+3,35%** dibanding SVM).
   * Mekanisme *Self-Attention* dua arah memungkinkan IndoBERT memahami makna kontekstual kalimat multi-topik warga (misal membedakan jalan rusak vs drainase) yang gagal ditangkap oleh RNN maupun SVM.
2. **Karakteristik Arsitektur RNN (BiGRU):**
   * Meraih **Macro F1-Score 94,68%** dan Akurasi 97,88%.
   * Jumlah parameter sangat efisien (**hanya ~940 ribu parameter** dibanding IndoBERT 124,5 juta parameter), dengan latensi inferensi yang sangat cepat (**3,42 ms**).
   * BiGRU sangat cocok dijadikan alternatif model untuk perangkat dengan keterbatasan memori komputasi (*edge devices* / peladen berdaya rendah).
3. **Kelayakan Implementasi Web:**
   * Kedua arsitektur Deep Learning (BiGRU 3,42 ms dan IndoBERT 12,36 ms) terbukti **sangat memenuhi ambang batas responsivitas sistem web interaktif (< 200 ms)**, sehingga siap diintegrasikan pada portal smart city Pemkot Makassar.